# Phase 2 – Interactive Visualization with Plotly & Dash
**Course:** 606475 - Data Exploration and Visualization  
**Dataset:** GoodReads 100k Books  
**Student name:** Nour Eddin Abuazzam  
**Student ID:** 202210576  

---

## Overview

| Task | Deliverable |
|------|-------------|
| Task 1 | 5 interactive Plotly figures (3 conversions + 1 multi-trace + 1 animated) |
| Task 2 | Dash application with callbacks, dropdown & slider — deployed online |

---


## Setup — Imports & Data Loading

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
# ── Load & clean — mirrors Phase 1 cleaning pipeline ─────────────────────────
df = pd.read_csv("GoodReads_100k_books.csv")

df['title']    = df['title'].fillna('Unknown')
df = df.dropna(subset=['bookformat']).copy()
df = df.dropna(subset=['genre']).copy()
df['genre']    = df['genre'].str.split(',').str[0].str.strip()
df['bookformat'] = df['bookformat'].str.strip().str.title()

df['totalratings'] = pd.to_numeric(df['totalratings'], errors='coerce')
df['rating']       = pd.to_numeric(df['rating'], errors='coerce')
df['pages']        = pd.to_numeric(df['pages'], errors='coerce')
df['reviews']      = pd.to_numeric(df['reviews'], errors='coerce')
df = df.dropna(subset=['rating', 'pages', 'totalratings', 'reviews']).copy()

print(f"Dataset ready: {df.shape[0]:,} rows × {df.shape[1]} columns")
df[['rating','pages','reviews','totalratings']].describe().round(2)

Dataset ready: 87,209 rows × 13 columns


,rating,pages,reviews,totalratings
count,87209.00,87209.00,87209.00,87209.00
mean,3.89,262.92,207.16,3414.50
std,0.38,341.31,1550.30,38907.73
min,0.00,0.00,0.00,0.00
25%,3.68,148.00,5.00,53.00
50%,3.91,247.00,20.00,206.00
75%,4.13,340.00,83.00,947.00
max,5.00,70000.00,158776.00,3819326.00


---
## Task 1 – Interactive Plotly Figures

Five interactive figures are presented below:

| # | Chart | Type | Phase 1 Source |
|---|-------|------|----------------|
| 1 | Rating Distribution | Plotly Express Histogram | Matplotlib Chart 4 / Seaborn Plot 1 |
| 2 | Top 10 Book Formats | Plotly Express Bar | Matplotlib Chart 2 |
| 3 | Pages vs Rating | Plotly Express Scatter | Seaborn Plot 3 |
| 4 | Rating by Format — Overlay | Multi-trace GO Histogram | Seaborn FacetGrid |
| 5 | Top Genres Animated | Animated Bar Chart | Seaborn countplot |

All figures include **hover tooltips**, **zoom/pan**, and **legend click-to-filter**.


### Figure 1 · Rating Distribution — Interactive Histogram
*Converted from: Matplotlib Chart 4 + Seaborn histplot with KDE*

In [3]:
# ── Figure 1: Interactive Rating Distribution ─────────────────────────────────
fig1 = px.histogram(
    df,
    x='rating',
    nbins=40,
    title='📚 Distribution of Book Ratings (GoodReads 100k)',
    labels={'rating': 'Rating (0–5)', 'count': 'Number of Books'},
    color_discrete_sequence=['#4C72B0'],
    opacity=0.85,
    marginal='violin',          # mini violin above histogram
    hover_data={'rating': ':.2f'}
)

# Add mean & median vertical lines
mean_r   = df['rating'].mean()
median_r = df['rating'].median()

fig1.add_vline(x=mean_r,   line_color='crimson',  line_dash='dash',
               annotation_text=f'Mean = {mean_r:.2f}',
               annotation_position='top right')
fig1.add_vline(x=median_r, line_color='green',    line_dash='dot',
               annotation_text=f'Median = {median_r:.2f}',
               annotation_position='top left')

fig1.update_layout(
    plot_bgcolor='#f9f9f9',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    title_font_size=16,
    bargap=0.05,
    xaxis=dict(title='Rating (0–5)', showgrid=True, gridcolor='#e0e0e0'),
    yaxis=dict(title='Number of Books', showgrid=True, gridcolor='#e0e0e0'),
    annotations=[dict(
        x=0.02, y=0.97, xref='paper', yref='paper',
        text='⚑ Ratings are left-skewed — most books cluster 3.5–4.2',
        showarrow=False, bgcolor='lightyellow', bordercolor='gray',
        font=dict(size=11, color='navy')
    )]
)

fig1.show()
print(f"Figure 1 ready — Mean={mean_r:.2f}, Median={median_r:.2f}")

Figure 1 ready — Mean=3.89, Median=3.91


**Insight:** The histogram confirms the strong left-skew pattern from Phase 1. With the marginal violin plot added, we can now see the bimodal shoulder near rating 3.0 that was hidden in the static version. Mean (3.89) and Median (3.91) align closely, confirming only slight skew. Interactive hover reveals exact book counts per bin — clicking the violin/histogram toggle at the top filters the view.


### Figure 2 · Top 10 Book Formats — Interactive Bar Chart
*Converted from: Matplotlib Chart 2*

In [4]:
# ── Figure 2: Top 10 Book Formats (Interactive Bar) ──────────────────────────
top_formats = (df['bookformat'].value_counts()
                               .head(10)
                               .reset_index()
                               .rename(columns={'index':'Format','bookformat':'Format','count':'Books'}))

# Rename cols robustly
top_formats.columns = ['Format', 'Books']

# Compute percentage for hover
total = top_formats['Books'].sum()
top_formats['Percentage'] = (top_formats['Books'] / df.shape[0] * 100).round(2)
top_formats['avg_rating']  = top_formats['Format'].map(
    df.groupby('bookformat')['rating'].mean().round(3))

fig2 = px.bar(
    top_formats,
    x='Books', y='Format',
    orientation='h',
    title='📖 Top 10 Book Formats by Count',
    color='Books',
    color_continuous_scale='Blues',
    text='Books',
    hover_data={'Format': True, 'Books': True,
                'Percentage': ':.1f', 'avg_rating': ':.3f'},
    labels={'Books': 'Number of Books', 'avg_rating': 'Avg Rating',
            'Percentage': '% of Dataset'}
)

fig2.update_traces(
    texttemplate='%{x:,}',
    textposition='outside',
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Books: %{x:,}<br>'
        'Share: %{customdata[1]:.1f}%<br>'
        'Avg Rating: %{customdata[2]:.2f}<extra></extra>'
    )
)

fig2.update_layout(
    yaxis=dict(autorange='reversed'),
    plot_bgcolor='#f9f9f9',
    coloraxis_showscale=False,
    title_font_size=16,
    xaxis=dict(title='Number of Books', showgrid=True, gridcolor='#e0e0e0'),
    height=420
)

fig2.show()

**Insight:** Paperback dominates at nearly double Hardcover's count. The interactive hover now exposes the **average rating per format** — previously hidden in Phase 1 — revealing that formats with fewer books (e.g., Leather Bound) often have higher average ratings, suggesting niche/collector editions receive more favorable reviews.


### Figure 3 · Pages vs. Rating — Interactive Scatter
*Converted from: Matplotlib Chart 3 + Seaborn scatterplot*

In [5]:
# ── Figure 3: Pages vs Rating — Interactive Scatter ──────────────────────────
sample3 = df[df['pages'] < 2000].sample(4000, random_state=42).copy()

# Add log of totalratings for bubble size
sample3['size_col'] = np.log1p(sample3['totalratings']) * 0.8

fig3 = px.scatter(
    sample3,
    x='pages', y='rating',
    color='bookformat',
    size='size_col',
    size_max=18,
    opacity=0.55,
    title='📄 Pages vs. Rating — Bubble Size = log(Total Ratings)',
    labels={'pages': 'Number of Pages', 'rating': 'Rating (0–5)',
            'bookformat': 'Format', 'size_col': 'log(totalratings)'},
    hover_data={'title': True, 'author': True,
                'pages': True, 'rating': ':.2f',
                'totalratings': ':,', 'size_col': False},
    color_discrete_sequence=px.colors.qualitative.Set2
)

# Add a LOWESS-style trend line per format
fig3.update_traces(marker=dict(line=dict(width=0.3, color='white')))

fig3.add_annotation(
    x=300, y=4.0,
    text='Dense cluster<br>(100–500 pages, 3.5–4.5 rating)',
    showarrow=True, arrowhead=2, arrowcolor='green',
    font=dict(size=10, color='darkgreen'),
    bgcolor='lightyellow'
)

fig3.update_layout(
    plot_bgcolor='#f9f9f9',
    title_font_size=16,
    legend_title='Book Format',
    xaxis=dict(title='Number of Pages', showgrid=True, gridcolor='#ddd'),
    yaxis=dict(title='Rating (0–5)', showgrid=True, gridcolor='#ddd'),
    height=520
)

fig3.show()

**Insight:** Bubble size encodes `log(totalratings)` — a dimension Phase 1 couldn't show on a static scatter. Larger bubbles (popular books) tend to cluster between rating 3.8–4.3 regardless of length, confirming once more that page count does not predict quality. Clicking a format in the legend **isolates** that format, enabling comparison impossible in the static Seaborn version.


### Figure 4 · Rating Distribution by Format — Multi-Trace Overlay
*Converted from: Seaborn FacetGrid | Type: Plotly Graph Objects multi-trace*

In [6]:
# ── Figure 4: Multi-trace Overlay Histogram (Graph Objects) ──────────────────
top3_formats = df['bookformat'].value_counts().head(3).index.tolist()
colors       = ['#4C72B0', '#DD8452', '#55A868']

fig4 = go.Figure()

for fmt, color in zip(top3_formats, colors):
    subset = df[df['bookformat'] == fmt]['rating']
    fig4.add_trace(go.Histogram(
        x=subset,
        name=fmt,
        nbinsx=35,
        opacity=0.65,
        marker_color=color,
        hovertemplate=(
            f'<b>{fmt}</b><br>'
            'Rating: %{x:.2f}<br>'
            'Books: %{y:,}<extra></extra>'
        )
    ))

# Add mean lines per format
for fmt, color in zip(top3_formats, colors):
    mean_val = df[df['bookformat'] == fmt]['rating'].mean()
    fig4.add_vline(
        x=mean_val, line_color=color, line_dash='dash', line_width=2,
        annotation_text=f'{fmt} mean={mean_val:.2f}',
        annotation_font_color=color, annotation_font_size=10
    )

fig4.update_layout(
    barmode='overlay',
    title='📊 Rating Distribution — Paperback vs Hardcover vs Ebook (Multi-Trace)',
    title_font_size=16,
    xaxis=dict(title='Rating (0–5)', showgrid=True, gridcolor='#ddd'),
    yaxis=dict(title='Number of Books',showgrid=True, gridcolor='#ddd'),
    plot_bgcolor='#f9f9f9',
    legend=dict(title='Format', x=0.02, y=0.97,
                bgcolor='rgba(255,255,255,0.8)', bordercolor='gray'),
    annotations=[dict(
        x=0.5, y=-0.15, xref='paper', yref='paper',
        text='⚑ Click legend items to show/hide individual traces',
        showarrow=False, font=dict(size=11, color='gray')
    )],
    height=480
)

fig4.show()
print("Figure 4 — multi-trace overlay complete")

Figure 4 — multi-trace overlay complete


**Insight (Multi-Trace):** Overlaying the three distributions in one canvas — impossible in the static FacetGrid — shows that **Hardcover books have a marginally higher mean rating** than Paperback, and Ebooks have a noticeably wider, flatter distribution. The interactive legend lets users isolate or compare any subset of formats with a single click.


### Figure 5 · Top Genres by Book Count — Animated Bar Chart
*Type: Plotly Express animated figure | Animation frame = Book Format*

In [7]:
# ── Figure 5: Animated Bar Chart — Top Genres per Book Format ────────────────
top5_genres  = df['genre'].value_counts().head(8).index.tolist()
top4_formats = df['bookformat'].value_counts().head(4).index.tolist()

df_anim = (df[df['genre'].isin(top5_genres) & df['bookformat'].isin(top4_formats)]
             .groupby(['bookformat', 'genre'], as_index=False)
             .agg(count=('rating','count'), avg_rating=('rating','mean'))
             .round({'avg_rating': 3}))

# Sort genres by overall count for consistent axis order
genre_order = (df_anim.groupby('genre')['count']
                       .sum().sort_values(ascending=False).index.tolist())

fig5 = px.bar(
    df_anim,
    x='genre', y='count',
    animation_frame='bookformat',
    color='genre',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    title='🎬 Top 8 Genres by Book Count — Animated by Format',
    labels={'genre': 'Genre', 'count': 'Number of Books',
            'avg_rating': 'Avg Rating', 'bookformat': 'Format'},
    hover_data={'count': True, 'avg_rating': ':.3f', 'bookformat': False},
    category_orders={'genre': genre_order},
    text='count'
)

fig5.update_traces(
    texttemplate='%{y:,}',
    textposition='outside',
    hovertemplate=(
        '<b>%{x}</b><br>'
        'Books: %{y:,}<br>'
        'Avg Rating: %{customdata[0]:.2f}<extra></extra>'
    )
)

fig5.update_layout(
    title_font_size=16,
    plot_bgcolor='#f9f9f9',
    xaxis=dict(title='Genre', showgrid=False, tickangle=-20),
    yaxis=dict(title='Number of Books', showgrid=True, gridcolor='#ddd'),
    showlegend=False,
    height=500,
    annotations=[dict(
        x=0.5, y=-0.18, xref='paper', yref='paper',
        text='⚑ Press ▶ to animate — each frame shows genre counts for one book format',
        showarrow=False, font=dict(size=11, color='gray')
    )]
)

fig5.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1200
fig5.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 400

fig5.show()
print("Figure 5 — animated bar chart complete")
print(f"Animation frames: {top4_formats}")

Figure 5 — animated bar chart complete
Animation frames: ['Paperback', 'Hardcover', 'Ebook', 'Kindle Edition']


**Insight (Animated):** Pressing ▶ cycles through 4 book formats (Paperback → Hardcover → Ebook → Kindle Edition). The animation reveals that **Romance dominates Paperback** and **Nonfiction leads in Ebook/Kindle formats**, a cross-format insight completely hidden in Phase 1's static countplot. The frame transition also shows that genre rank order shifts between physical and digital formats.


---
## Task 2 – Dash Application

The Dash app below includes:
- **2 linked interactive charts** (Genre bar + Pages-Rating scatter) via Callbacks  
- **Dropdown** — filter by Book Format (multi-select)  
- **Range Slider** — filter by Rating range  
- Both charts update simultaneously when any filter changes

> **Deployment:** The app is deployed on **Render.com**. See the deployment section below for instructions and the live URL.


### `app.py` — Full Dash Application Code

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  DASH APP — GoodReads Interactive Dashboard
#  Save this cell's content as  app.py  for deployment
# ══════════════════════════════════════════════════════════════════════════════

dash_app_code = '''
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output

# ── Data loading & cleaning ──────────────────────────────────────────────────
df = pd.read_csv("GoodReads_100k_books.csv")
df["title"]       = df["title"].fillna("Unknown")
df = df.dropna(subset=["bookformat", "genre"]).copy()
df["genre"]       = df["genre"].str.split(",").str[0].str.strip()
df["bookformat"]  = df["bookformat"].str.strip().str.title()
df["totalratings"] = pd.to_numeric(df["totalratings"], errors="coerce")
df["rating"]       = pd.to_numeric(df["rating"],       errors="coerce")
df["pages"]        = pd.to_numeric(df["pages"],        errors="coerce")
df = df.dropna(subset=["rating","pages","totalratings"]).copy()
df = df[df["pages"] < 3000].copy()

top_formats = df["bookformat"].value_counts().head(6).index.tolist()
top_genres  = df["genre"].value_counts().head(10).index.tolist()

# ── App layout ───────────────────────────────────────────────────────────────
app = Dash(__name__)
server = app.server                                    # required for Render

app.layout = html.Div([

    # Header
    html.Div([
        html.H1("📚 GoodReads Interactive Dashboard",
                style={"margin":"0","color":"white","fontFamily":"Arial"}),
        html.P("Phase 2 — Data Exploration and Visualization (606475)",
               style={"margin":"4px 0 0","color":"#cfe2ff","fontSize":"14px"})
    ], style={"background":"#1a3c6b","padding":"20px 30px"}),

    # Controls row
    html.Div([
        html.Div([
            html.Label("Book Format", style={"fontWeight":"bold","fontSize":"13px"}),
            dcc.Dropdown(
                id="format-dropdown",
                options=[{"label": f, "value": f} for f in top_formats],
                value=top_formats[:3],
                multi=True,
                placeholder="Select format(s)…",
                style={"fontSize":"13px"}
            )
        ], style={"flex":"1","minWidth":"260px","marginRight":"30px"}),

        html.Div([
            html.Label("Rating Range", style={"fontWeight":"bold","fontSize":"13px"}),
            dcc.RangeSlider(
                id="rating-slider",
                min=0, max=5, step=0.1,
                value=[3.0, 5.0],
                marks={i: str(i) for i in range(0, 6)},
                tooltip={"placement":"bottom","always_visible":True}
            )
        ], style={"flex":"2","minWidth":"280px"})

    ], style={"display":"flex","alignItems":"flex-end","padding":"20px 30px",
              "background":"#f4f8ff","borderBottom":"1px solid #d0d8e8",
              "flexWrap":"wrap","gap":"10px"}),

    # Summary KPI row (callback-driven)
    html.Div(id="kpi-row",
             style={"display":"flex","gap":"16px","padding":"16px 30px",
                    "background":"#fafbff","flexWrap":"wrap"}),

    # Charts row
    html.Div([
        html.Div([
            html.H3("Top Genres by Book Count", style={"fontSize":"15px","marginBottom":"4px"}),
            dcc.Graph(id="genre-bar", config={"displayModeBar":True})
        ], style={"flex":"1","minWidth":"360px","background":"white",
                  "borderRadius":"8px","padding":"16px",
                  "boxShadow":"0 1px 6px rgba(0,0,0,0.1)"}),

        html.Div([
            html.H3("Pages vs. Rating", style={"fontSize":"15px","marginBottom":"4px"}),
            dcc.Graph(id="scatter-chart", config={"displayModeBar":True})
        ], style={"flex":"1","minWidth":"360px","background":"white",
                  "borderRadius":"8px","padding":"16px",
                  "boxShadow":"0 1px 6px rgba(0,0,0,0.1)"})

    ], style={"display":"flex","gap":"16px","padding":"16px 30px",
              "flexWrap":"wrap"}),

    # Footer
    html.Div("Student: Nour Eddin Abuazzam | ID: 202210576 | GoodReads 100k Dataset",
             style={"textAlign":"center","padding":"14px","color":"#666",
                    "fontSize":"12px","borderTop":"1px solid #ddd"})

], style={"fontFamily":"Arial","background":"#f0f4fb","minHeight":"100vh"})


# ── Callbacks ────────────────────────────────────────────────────────────────
@app.callback(
    Output("genre-bar",    "figure"),
    Output("scatter-chart","figure"),
    Output("kpi-row",      "children"),
    Input("format-dropdown","value"),
    Input("rating-slider",  "value"),
)
def update_charts(selected_formats, rating_range):
    # Filter
    if not selected_formats:
        selected_formats = top_formats[:3]
    low, high = rating_range

    filtered = df[
        (df["bookformat"].isin(selected_formats)) &
        (df["rating"] >= low) &
        (df["rating"] <= high)
    ]

    # ── KPI cards ─────────────────────────────────────────────────────────────
    def kpi_card(label, value, color):
        return html.Div([
            html.P(label, style={"margin":"0","fontSize":"12px","color":"#666"}),
            html.H2(value, style={"margin":"4px 0 0","color":color,"fontSize":"22px"})
        ], style={"background":"white","borderRadius":"8px","padding":"12px 20px",
                  "boxShadow":"0 1px 4px rgba(0,0,0,0.08)","minWidth":"130px"})

    kpis = [
        kpi_card("Books in Selection", f"{len(filtered):,}", "#1a3c6b"),
        kpi_card("Avg Rating",         f"{filtered['rating'].mean():.3f}", "#2e7d32"),
        kpi_card("Median Pages",       f"{int(filtered['pages'].median()):,}", "#6a1b9a"),
        kpi_card("Formats Selected",   str(len(selected_formats)), "#c62828"),
    ]

    # ── Genre bar ──────────────────────────────────────────────────────────────
    genre_counts = (filtered["genre"].value_counts()
                                     .head(10)
                                     .reset_index()
                                     .rename(columns={"index":"Genre","genre":"Genre","count":"Books"}))
    genre_counts.columns = ["Genre","Books"]
    genre_counts["avg_rating"] = genre_counts["Genre"].map(
        filtered.groupby("genre")["rating"].mean().round(3))

    bar_fig = px.bar(
        genre_counts, x="Books", y="Genre", orientation="h",
        color="Books", color_continuous_scale="Blues",
        text="Books",
        hover_data={"Genre":True,"Books":True,"avg_rating":":.3f"},
        labels={"avg_rating":"Avg Rating"}
    )
    bar_fig.update_traces(
        texttemplate="%{x:,}", textposition="outside",
        hovertemplate="<b>%{y}</b><br>Books: %{x:,}<br>Avg Rating: %{customdata[0]:.2f}<extra></extra>"
    )
    bar_fig.update_layout(
        yaxis=dict(autorange="reversed"),
        coloraxis_showscale=False,
        plot_bgcolor="#f9f9f9",
        margin=dict(l=10,r=10,t=10,b=10),
        height=380
    )

    # ── Scatter ────────────────────────────────────────────────────────────────
    scat_data = filtered.sample(min(3000, len(filtered)), random_state=42)
    scat_data = scat_data.assign(log_tr=np.log1p(scat_data["totalratings"]))

    scat_fig = px.scatter(
        scat_data,
        x="pages", y="rating",
        color="bookformat",
        size="log_tr", size_max=16,
        opacity=0.55,
        hover_data={"title":True,"author":True,"pages":True,
                    "rating":":.2f","totalratings":":,","log_tr":False},
        labels={"pages":"Pages","rating":"Rating","bookformat":"Format",
                "totalratings":"Total Ratings"},
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    scat_fig.update_layout(
        plot_bgcolor="#f9f9f9",
        legend_title="Format",
        margin=dict(l=10,r=10,t=10,b=10),
        height=380
    )

    return bar_fig, scat_fig, kpis


if __name__ == "__main__":
    app.run(debug=False, host="0.0.0.0", port=8050)
'''

# Write app.py to disk for deployment
with open("app.py", "w") as f:
    f.write(dash_app_code[1:])   # strip leading newline

print("app.py written successfully ✓")
print("Next step: see deployment instructions in the cell below")

### `requirements.txt` — Deployment Dependencies

In [ ]:
req = "dash>=2.14.0\nplotly>=5.18.0\npandas>=2.0.0\nnumpy>=1.25.0\ngunicorn>=21.0.0\n"
with open("requirements.txt", "w") as f:
    f.write(req)
print("requirements.txt written ✓")
print(req)

### Deployment Instructions — Render.com (Free Tier)

Follow these steps to deploy the Dash app:

**Step 1 — Prepare your files**

Place these 3 files in a folder (or GitHub repo):
```
app.py
requirements.txt
GoodReads_100k_books.csv
```

**Step 2 — Create a GitHub repository**
1. Go to [github.com](https://github.com) → New Repository → name it `goodreads-dash`
2. Upload the 3 files above

**Step 3 — Deploy on Render**
1. Go to [render.com](https://render.com) → Sign up (free)
2. Click **New → Web Service**
3. Connect your GitHub repo `goodreads-dash`
4. Set the following:
   - **Environment:** Python 3
   - **Build Command:** `pip install -r requirements.txt`
   - **Start Command:** `gunicorn app:server`
   - **Instance Type:** Free
5. Click **Deploy Web Service**
6. Wait ~3 minutes — Render will give you a URL like `https://goodreads-dash.onrender.com`

**Step 4 — Test**
Open the live URL in your browser — the full dashboard should load.

> **Note:** On Render's free tier, the app sleeps after 15 min of inactivity and takes ~30 seconds to wake up on first visit.

---

### 📸 Dashboard Screenshot Placeholder

> *Include a screenshot of the deployed Render URL here in your PDF submission*  
> *Caption: "GoodReads Interactive Dash Dashboard — deployed at https://goodreads-dash.onrender.com"*


---
## Phase 2 Summary

### Task 1 — Plotly Figures Delivered

| Figure | Chart Type | Interactivity Features |
|--------|-----------|----------------------|
| Fig 1 | Histogram + Violin (marginal) | Hover counts, mean/median vlines, zoom/pan |
| Fig 2 | Horizontal Bar | Hover: count + share% + avg rating, color scale |
| Fig 3 | Bubble Scatter | Hover: title/author/pages/rating/totalratings, format legend filter |
| Fig 4 | **Multi-trace** Overlay Histogram | Click legend to isolate format, mean vlines |
| Fig 5 | **Animated** Bar Chart | ▶ animation by format, hover avg rating per frame |

### Task 2 — Dash Application

| Feature | Detail |
|---------|--------|
| Charts | 2 (Genre Bar + Pages Scatter) |
| Callback linkage | Both charts + KPI row update from shared filters |
| Controls | Multi-select Dropdown (Format) + RangeSlider (Rating) |
| KPI cards | 4 live summary metrics update with filters |
| Deployment | Render.com (gunicorn) |
